# Florence-2 MTL Fine-Tuning — Kaggle Notebook

## Setup
1. **Add your dataset** as a Kaggle Dataset input.  
   Update `paths.data_root` in `config.yaml` to match the mount path (e.g. `/kaggle/input/your-dataset-name`).
2. **Fill in secrets** in `config.yaml`:  
   - `wandb.api_key` — your Weights & Biases API key  
   - `huggingface.api_key` + `huggingface.username` — only if you want to push to HF Hub
3. **Resume**: set `resume.enabled: true` in `config.yaml` to resume from the latest checkpoint automatically.
4. Run the cells below in order.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
# Run once. You can skip on subsequent resume runs if kernel is the same.
!pip install -q peft wandb transformers==4.49.0 accelerate

In [ ]:
# ── Cell 2: Clone / copy the training repo ────────────────────────────────
# Option A — if you're using the code from a Kaggle Dataset input:
import shutil, os

REPO_INPUT = "/kaggle/input/fyp-mtl-gi-vqa-code"   # <- adjust to your code dataset name
REPO_DEST  = "/kaggle/working/repo"

if os.path.exists(REPO_INPUT):                      # Code dataset is attached
    shutil.copytree(REPO_INPUT, REPO_DEST, dirs_exist_ok=True)
    print(f"[OK] Repo copied to {REPO_DEST}")
else:
    print("[INFO] No code dataset found — make sure the repo is in the working dir or attached.")

# Option B — if you're cloning from GitHub (uncomment & fill in your repo URL):
# !git clone https://github.com/<your-user>/<your-repo>.git {REPO_DEST}

In [ ]:
# ── Cell 3: (Optional) Edit config inline ─────────────────────────────────
# You can patch config.yaml values here without editing the file directly.
# This is useful for quick W&B key injection via Kaggle Secrets.

import yaml, os

CONFIG_PATH = "/kaggle/working/repo/kaggle_training/config.yaml"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# ---- Inject Kaggle Secrets (recommended) ---------------------------------
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

try:
    cfg["wandb"]["api_key"]       = secrets.get_secret("WANDB_API_KEY")
except Exception:
    print("[Secrets] WANDB_API_KEY not found in Kaggle Secrets — using config value.")

try:
    cfg["huggingface"]["api_key"] = secrets.get_secret("HF_TOKEN")
except Exception:
    print("[Secrets] HF_TOKEN not found in Kaggle Secrets — using config value.")

# ---- Write patched config back -------------------------------------------
with open(CONFIG_PATH, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print("[OK] Config ready.")
print(yaml.dump(cfg, default_flow_style=False))

In [ ]:
# ── Cell 4: Run training ──────────────────────────────────────────────────
import subprocess, sys

REPO_DEST   = "/kaggle/working/repo"   # must match Cell 2
CONFIG_PATH = f"{REPO_DEST}/kaggle_training/config.yaml"
TRAIN_SCRIPT= f"{REPO_DEST}/kaggle_training/kaggle_train.py"

result = subprocess.run(
    [sys.executable, TRAIN_SCRIPT, "--config", CONFIG_PATH],
    check=False          # set True to raise on non-zero exit
)

print(f"\n[Done] Process exited with code: {result.returncode}")